# ProtoEHR — Fusion (standalone, self-contained)

Run AFTER both `global_branch_hpo.ipynb` and `local_branch_hpo.ipynb`
have finished and printed their `final_cfg` dicts. Paste both dicts into
the first code cell below the shared setup, then run top to bottom.
Retrains both branches at full budget, freezes them, fuses, and reports
held-out test metrics -- test is touched exactly once, in the final cell.


In [0]:
%pip install torch optuna

In [0]:

import os, time, copy, json, math
from concurrent.futures import ProcessPoolExecutor, as_completed
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, brier_score_loss, balanced_accuracy_score
)

CLASS_NAMES   = ["control", "pe_ghtn", "gdm"]
OUTCOME_NAMES = ["pe_ghtn", "gdm"]

# ---- shared model/loss hyperparameters (branch-specific overrides layered on top) ----
BASE_CFG = {
    "flags_path":     "catalog.schema.step7_filtered_cohort_with_apos",
    "output_dir":     "./proto_runs",
    "embedding_dim":  768,
    "adapter_hidden": 256,
    "adapter_out":    64,
    "n_classes":      3,

    "prototypes_per_class": {"control": 6, "pe_ghtn": 8, "gdm": 5},

    "lr": 2e-3,
    "weight_decay": 1e-4,
    "batch_size": 512,
    "max_epochs": 200,
    "patience": 10,

    "lambda_clst": 4e-4, "lambda_sep": 4e-4, "lambda_div": 1.0,
    "lambda_cntrst": 300.0, "lambda_covar": 0.5,
    "zero_diagonal_blocks": True,

    "use_jscl_pretrain": False,

    "max_iterations": 5, "iter_patience": 2, "lr_decay": 0.5,
    "random_seed": 42,
    "run_id": "run_default",
}

GLOBAL_CFG = copy.deepcopy(BASE_CFG)
GLOBAL_CFG["embeddings_path"] = "catalog.schema.patient_clmbr_embeddings_new_v5"

LOCAL_CFG = copy.deepcopy(BASE_CFG)
LOCAL_CFG["embeddings_path"] = "catalog.schema.patient_clmbr_embeddings_new_v5_rolling_trimester_delta"
LOCAL_CFG["topk_pool_k"] = 5
LOCAL_CFG["max_train_patients_per_stratum"] = None  # set an int to cap memory, see Section 2

os.makedirs(BASE_CFG["output_dir"], exist_ok=True)


def total_prototypes(cfg):
    return sum(cfg["prototypes_per_class"][c] for c in CLASS_NAMES)


In [0]:

def build_person_split(cfg):
    """Person-level 80/10/10 stratified split on (control, pe_ghtn, gdm)
    combo. This is the SINGLE source of truth for split membership --
    both branches call this with the same cfg['flags_path'] and
    cfg['random_seed'] and get identical (pid_train, pid_val, pid_test)."""
    flags_df = spark.table(cfg["flags_path"]).select(
        "person_id", "date_of_birth", "recorded_episode_start", "inferred_episode_start",
        "has_pe", "has_inferred_pe", "has_ghtn", "has_gdm", "is_control"
    ).toPandas()
    flags_df["person_id"] = flags_df["person_id"].astype(str)

    flags_df["y_control"] = flags_df["is_control"].astype(int)
    flags_df["y_pe_ghtn"] = ((flags_df["has_pe"] == 1) | (flags_df["has_inferred_pe"] == 1)
                              | (flags_df["has_ghtn"] == 1)).astype(int)
    flags_df["y_gdm"] = flags_df["has_gdm"].astype(int)

    # age at episode_start -- computed directly (date_of_birth vs episode_start),
    # NOT from a condition-code proxy -- used later as the fusion covariate
    for c in ["date_of_birth", "recorded_episode_start", "inferred_episode_start"]:
        flags_df[c] = pd.to_datetime(flags_df[c])
    flags_df["episode_start"] = flags_df[["recorded_episode_start", "inferred_episode_start"]].min(axis=1, skipna=True)
    flags_df["age"] = (flags_df["episode_start"] - flags_df["date_of_birth"]).dt.days / 365.25

    combo = (flags_df["y_control"]*4 + flags_df["y_pe_ghtn"]*2 + flags_df["y_gdm"]).astype(int)
    pid_tv, pid_test = train_test_split(
        flags_df["person_id"].values, test_size=0.10, stratify=combo, random_state=cfg["random_seed"]
    )
    combo_tv = combo[flags_df["person_id"].isin(pid_tv)].values
    pid_train, pid_val = train_test_split(
        pid_tv, test_size=0.10/0.90, stratify=combo_tv, random_state=cfg["random_seed"]
    )
    return {"pid_train": set(pid_train), "pid_val": set(pid_val), "pid_test": set(pid_test),
            "flags_df": flags_df}


def load_global_data(cfg, split, device):
    """Flat, single-embedding-per-patient table -> (X, Y, pid, age) per split."""
    df = spark.table(cfg["embeddings_path"]).toPandas()
    df["patient_id"] = df["patient_id"].astype(str)
    label_cols = ["y_control", "y_pe_ghtn", "y_gdm"]
    df = df.merge(split["flags_df"][["person_id"] + label_cols + ["age"]],
                  left_on="patient_id", right_on="person_id", how="inner")

    emb_cols = sorted([c for c in df.columns if c.startswith("emb_")], key=lambda x: int(x.split("_")[1]))
    assert len(emb_cols) == cfg["embedding_dim"]

    out = {}
    for name, pid_set in [("train", split["pid_train"]), ("val", split["pid_val"]), ("test", split["pid_test"])]:
        sub = df[df["patient_id"].isin(pid_set)]
        out[f"X_{name}"] = sub[emb_cols].values.astype(np.float32)
        out[f"Y_{name}"] = sub[label_cols].values.astype(np.float32)
        out[f"pid_{name}"] = sub["patient_id"].values
        out[f"age_{name}"] = sub["age"].values.astype(np.float32)

    pos_freq = out["Y_train"].mean(axis=0).clip(0.03, 0.97)
    out["class_weights"] = torch.tensor((1 - pos_freq) / pos_freq, dtype=torch.float32).to(device)
    print(f"[global] train={len(out['X_train'])} val={len(out['X_val'])} test={len(out['X_test'])}")
    return out


def load_local_data(cfg, split, device):
    """Long-format rolling-delta table -> row-level (X, Y-broadcast, pid, age)
    per split, dropping NaN-embedding rows (delta not yet computable / source
    window unavailable). Uses the chunked patient-filtered pull pattern
    established earlier in this project so peak memory stays bounded, with
    an optional per-stratum training cap (cfg['max_train_patients_per_stratum']).
    """
    flags_df = split["flags_df"]
    label_cols = ["y_control", "y_pe_ghtn", "y_gdm"]

    avail_pids = spark.sql(
        f"SELECT DISTINCT patient_id FROM {cfg['embeddings_path']} WHERE unavailable = 0"
    ).toPandas()
    avail_pids["patient_id"] = avail_pids["patient_id"].astype(str)
    patient_labels = avail_pids.merge(flags_df, left_on="patient_id", right_on="person_id", how="inner")

    def pull_for(pid_set, cap_per_stratum=None):
        pids = patient_labels[patient_labels["patient_id"].isin(pid_set)].copy()
        if cap_per_stratum is not None:
            combo = (pids["y_control"]*4 + pids["y_pe_ghtn"]*2 + pids["y_gdm"]).astype(int).values
            rng = np.random.RandomState(cfg["random_seed"])
            keep = []
            for stratum in np.unique(combo):
                idxs = np.where(combo == stratum)[0]
                if len(idxs) > cap_per_stratum:
                    idxs = rng.choice(idxs, size=cap_per_stratum, replace=False)
                keep.extend(idxs.tolist())
            pids = pids.iloc[sorted(keep)]

        chunk_size, chunks = 2000, []
        ids = pids["patient_id"].tolist()
        for i in range(0, len(ids), chunk_size):
            batch_ids = ids[i:i + chunk_size]
            id_list_sql = ", ".join(f"'{pid}'" for pid in batch_ids)
            chunk_df = spark.sql(f'''
                SELECT * FROM {cfg['embeddings_path']}
                WHERE patient_id IN ({id_list_sql}) AND unavailable = 0
            ''').toPandas()
            emb_cols = sorted([c for c in chunk_df.columns if c.startswith("emb_")], key=lambda x: int(x.split("_")[1]))
            chunk_df[emb_cols] = chunk_df[emb_cols].astype(np.float32)
            chunk_df["patient_id"] = chunk_df["patient_id"].astype(str)
            chunks.append(chunk_df)
        rows = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()
        if rows.empty:
            return rows, []
        emb_cols = sorted([c for c in rows.columns if c.startswith("emb_")], key=lambda x: int(x.split("_")[1]))
        nan_mask = rows[emb_cols].isna().any(axis=1)
        rows = rows.loc[~nan_mask].reset_index(drop=True)
        rows = rows.merge(pids[["patient_id"] + label_cols + ["age"]], on="patient_id", how="inner")
        return rows, emb_cols

    train_rows, emb_cols = pull_for(split["pid_train"], cfg.get("max_train_patients_per_stratum"))
    val_rows, _ = pull_for(split["pid_val"])
    test_rows, _ = pull_for(split["pid_test"])

    out = {}
    for name, rows in [("train", train_rows), ("val", val_rows), ("test", test_rows)]:
        out[f"X_{name}"] = rows[emb_cols].values.astype(np.float32)
        out[f"Y_{name}"] = rows[label_cols].values.astype(np.float32)
        out[f"pid_{name}"] = rows["patient_id"].values
        out[f"age_{name}"] = rows["age"].values.astype(np.float32)
        out[f"window_name_{name}"] = rows["window_name"].values if "window_name" in rows.columns else np.array([None]*len(rows))

    pos_freq = out["Y_train"].mean(axis=0).clip(0.03, 0.97)
    out["class_weights"] = torch.tensor((1 - pos_freq) / pos_freq, dtype=torch.float32).to(device)
    print(f"[local]  train={len(out['X_train'])} rows / {len(set(out['pid_train']))} patients | "
          f"val={len(out['X_val'])} rows | test={len(out['X_test'])} rows")
    return out


In [0]:

def class_offsets(prototypes_per_class):
    offsets, start = {}, 0
    for name in CLASS_NAMES:
        k = prototypes_per_class[name]
        offsets[name] = (start, start + k)
        start += k
    return offsets, start


class AdapterMLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
            nn.BatchNorm1d(out_dim, affine=False),
        )
    def forward(self, x):
        return self.net(x)


class PrototypeLayer(nn.Module):
    def __init__(self, prototypes_per_class, prototype_dim, scale, verbose=True):
        super().__init__()
        self.prototypes_per_class = prototypes_per_class
        self.offsets, self.n_prototypes = class_offsets(prototypes_per_class)
        self.prototype_dim = prototype_dim
        self.scale = scale
        self.prototypes = nn.Parameter(torch.randn(self.n_prototypes, prototype_dim) * 0.1)

        assignment = torch.zeros(len(CLASS_NAMES), self.n_prototypes)
        for c, name in enumerate(CLASS_NAMES):
            s, e = self.offsets[name]
            assignment[c, s:e] = 1.0
        self.register_buffer("assignment", assignment)
        if verbose:
            print(f"  prototypes: {self.n_prototypes} total -> " +
                  ", ".join(f"{n}={self.offsets[n][1]-self.offsets[n][0]}" for n in CLASS_NAMES))

    def forward(self, z):
        z_norm = F.normalize(z, dim=1)
        p_norm = F.normalize(self.prototypes, dim=1)
        return self.scale * (z_norm @ p_norm.T)


class ProtoEHRNet(nn.Module):
    def __init__(self, cfg, scale, verbose=True):
        super().__init__()
        self.adapter = AdapterMLP(cfg["embedding_dim"], cfg["adapter_hidden"], cfg["adapter_out"])
        self.proto_layer = PrototypeLayer(cfg["prototypes_per_class"], cfg["adapter_out"], scale, verbose)
        self.classifier = nn.Linear(self.proto_layer.n_prototypes, cfg["n_classes"], bias=True)

    def forward(self, x):
        z = self.adapter(x)
        sim = self.proto_layer(z)
        logits = self.classifier(sim)
        return logits, sim, z


In [0]:

def bce_loss(logits, labels, class_weights):
    return F.binary_cross_entropy_with_logits(logits, labels, pos_weight=class_weights, reduction="mean")

def clustering_loss(sim, labels, assignment):
    pos_mask = (labels @ assignment).clamp(0, 1).bool()
    masked = sim.masked_fill(~pos_mask, -1e9)
    has_pos = pos_mask.any(dim=1)
    return -masked[has_pos].max(dim=1).values.mean() if has_pos.any() else sim.new_tensor(0.0)

def separation_loss(sim, labels, assignment):
    neg_mask = ((labels @ assignment) == 0)
    masked = sim.masked_fill(~neg_mask, -1e9)
    has_neg = neg_mask.any(dim=1)
    return masked[has_neg].max(dim=1).values.mean() if has_neg.any() else sim.new_tensor(0.0)

def orthogonality_loss(prototypes):
    p = F.normalize(prototypes, dim=1)
    gram = p @ p.T
    I = torch.eye(gram.size(0), device=gram.device)
    return torch.norm(gram - I, p="fro") ** 2

def build_jaccard_matrix(Y_train, prototypes_per_class, zero_diagonal_blocks=True):
    n_classes = len(CLASS_NAMES)
    C_class = np.zeros((n_classes, n_classes))
    for c1 in range(n_classes):
        for c2 in range(n_classes):
            inter = ((Y_train[:, c1] == 1) & (Y_train[:, c2] == 1)).sum()
            union = ((Y_train[:, c1] == 1) | (Y_train[:, c2] == 1)).sum()
            C_class[c1, c2] = inter / union if union > 0 else 0.0
    if zero_diagonal_blocks:
        np.fill_diagonal(C_class, 0.0)
    offsets, P = class_offsets(prototypes_per_class)
    C_proto = np.zeros((P, P))
    for c1, n1 in enumerate(CLASS_NAMES):
        for c2, n2 in enumerate(CLASS_NAMES):
            rs, re = offsets[n1]; cs, ce = offsets[n2]
            C_proto[rs:re, cs:ce] = C_class[c1, c2]
    return torch.tensor(C_proto, dtype=torch.float32)

def contrastive_loss(prototypes, C):
    p = F.normalize(prototypes, dim=1)
    S = p @ p.T
    C_comp = 1.0 - C
    pos_term = (C * S).sum() / C.sum().clamp(min=1e-8)
    neg_term = (C_comp * S).sum() / C_comp.sum().clamp(min=1e-8)
    return -(1.0 / prototypes.size(0) ** 0.5) * (pos_term - neg_term)

def covariance_loss(z, eps=1e-4):
    z = z - z.mean(dim=0, keepdim=True)
    n, d = z.shape
    cov = (z.T @ z) / (n - 1)
    off_diag = cov - torch.diag(torch.diagonal(cov))
    return (off_diag ** 2).sum() / d


In [0]:

def _compute_metric_suite(y_true, y_prob, class_names=CLASS_NAMES, outcome_names=OUTCOME_NAMES):
    results = {}
    y_pred = (y_prob >= 0.5).astype(int)
    for c, name in enumerate(class_names):
        yt, yp, ypred = y_true[:, c], y_prob[:, c], y_pred[:, c]
        n_pos, n_neg = int(yt.sum()), int((1 - yt).sum())
        results[f"auroc_{name}"] = float(roc_auc_score(yt, yp)) if n_pos > 0 and n_neg > 0 else float("nan")
        results[f"auprc_{name}"] = float(average_precision_score(yt, yp)) if n_pos > 0 and n_neg > 0 else float("nan")
        results[f"f1_{name}"] = float(f1_score(yt, ypred, zero_division=0)) if n_pos > 0 else float("nan")
        results[f"brier_{name}"] = float(brier_score_loss(yt, yp))
        results[f"bacc_{name}"] = float(balanced_accuracy_score(yt, ypred)) if n_pos > 0 and n_neg > 0 else float("nan")

    def macro(prefix):
        vals = [results[f"{prefix}_{n}"] for n in class_names if not np.isnan(results[f"{prefix}_{n}"])]
        return float(np.mean(vals)) if vals else float("nan")

    results["auroc_macro"] = macro("auroc")
    results["auprc_macro"] = macro("auprc")
    results["f1_macro"] = macro("f1")
    results["bacc_macro"] = macro("bacc")
    results["brier_macro"] = macro("brier")

    out_auroc = [results[f"auroc_{n}"] for n in outcome_names if not np.isnan(results[f"auroc_{n}"])]
    results["auroc_macro_outcomes_only"] = float(np.mean(out_auroc)) if out_auroc else float("nan")
    out_auprc = [results[f"auprc_{n}"] for n in outcome_names if not np.isnan(results[f"auprc_{n}"])]
    results["auprc_macro_outcomes_only"] = float(np.mean(out_auprc)) if out_auprc else float("nan")

    results["_primary"] = results["auroc_macro"]
    return results


def evaluate_all(model, X, Y, device, return_probs=False):
    """GLOBAL branch: row-level (one embedding per patient already)."""
    model.eval()
    with torch.no_grad():
        logits, _, _ = model(torch.tensor(X, dtype=torch.float32).to(device))
        probs = torch.sigmoid(logits).cpu().numpy()
    results = _compute_metric_suite(Y, probs)
    return (results, probs) if return_probs else results


@torch.no_grad()
def topk_pool_patient_probs(model, X, pid, device, k=5, batch_size=512):
    """LOCAL branch: row-level forward pass, then top-k pooling of
    per-window prototype similarities into one pooled vector per
    patient, classified once per patient."""
    model.eval()
    all_sim = []
    for start in range(0, len(X), batch_size):
        xb = torch.tensor(X[start:start + batch_size], dtype=torch.float32).to(device)
        z = model.adapter(xb)
        all_sim.append(model.proto_layer(z).cpu().numpy())
    sim_all = np.concatenate(all_sim, axis=0)
    n_p = sim_all.shape[1]
    proto_cols = [f"p{i}" for i in range(n_p)]
    df = pd.DataFrame(sim_all, columns=proto_cols)
    df["pid"] = np.asarray(pid)

    def pool(g):
        vals = g[proto_cols].values
        k_eff = min(k, vals.shape[0])
        return pd.Series(np.sort(vals, axis=0)[-k_eff:, :].mean(axis=0), index=proto_cols)

    pooled = df.groupby("pid", sort=False).apply(pool)
    pooled_pid = pooled.index.values
    pooled_sim = pooled.values.astype(np.float32)
    logits = model.classifier(torch.tensor(pooled_sim, dtype=torch.float32).to(device))
    return pooled_pid, torch.sigmoid(logits).cpu().numpy(), pooled_sim


def evaluate_patient_level(model, X, Y, pid, device, k=5, return_probs=False):
    pooled_pid, probs, _ = topk_pool_patient_probs(model, X, pid, device, k=k)
    y_df = pd.DataFrame(Y, columns=[f"y{i}" for i in range(Y.shape[1])])
    y_df["pid"] = np.asarray(pid)
    y_patient = y_df.drop_duplicates("pid").set_index("pid").loc[pooled_pid].values.astype(np.float32)
    results = _compute_metric_suite(y_patient, probs)
    return (results, probs, pooled_pid) if return_probs else results


def prototype_diagnostics(model):
    offsets, _ = class_offsets(model.proto_layer.prototypes_per_class)
    with torch.no_grad():
        p = F.normalize(model.proto_layer.prototypes.cpu(), dim=1)
        gram = (p @ p.T).numpy()
    concentrations = []
    for name in CLASS_NAMES:
        s, e = offsets[name]; k = e - s
        if k <= 1:
            continue
        block = gram[s:e, s:e]
        concentrations.append((block.sum() - np.trace(block)) / (k * (k - 1)))
    return {"proto_concentration": float(np.mean(concentrations)) if concentrations else float("nan")}


In [0]:

def project_prototypes(model, X_train, Y_train, pid_train, device, verbose=True):
    """Cross-class exclusion (a patient claimed by one class's prototype
    cannot also be claimed by another class's) + a SOFT, PER-SLOT
    comorbidity preference"""
    model.eval()
    with torch.no_grad():
        all_z = []
        for start in range(0, len(X_train), 512):
            xb = torch.tensor(X_train[start:start+512], dtype=torch.float32).to(device)
            all_z.append(model.adapter(xb))
        all_z = torch.cat(all_z, dim=0)
        z_norm = F.normalize(all_z, dim=1)
        p_norm_all = F.normalize(model.proto_layer.prototypes, dim=1)
        scale = model.proto_layer.scale
        n_train = len(X_train)
        used_mask = torch.zeros(n_train, dtype=torch.bool, device=device)
        n_labels_positive = torch.tensor(Y_train.sum(axis=1), device=device)
        purity_mask = (n_labels_positive == 1)

        proj = {}
        for name in CLASS_NAMES:
            s, e = model.proto_layer.offsets[name]
            class_idx = CLASS_NAMES.index(name)
            class_mask = torch.tensor(Y_train[:, class_idx] == 1, device=device)
            if not class_mask.any():
                continue
            p_block = p_norm_all[s:e]
            n_prototypes_this_class = e - s

            for k in range(n_prototypes_this_class):
                p_idx = s + k
                # PREFER pure (single-label) unclaimed patients; fall back to
                # ANY unclaimed same-class patient only if the pure pool is
                # exhausted for this class at this point in the loop.
                pure_eligible = class_mask & (~used_mask) & purity_mask
                eligible_mask = pure_eligible if pure_eligible.any() else (class_mask & (~used_mask))
                if not eligible_mask.any():
                    if verbose:
                        print(f"  WARNING: class '{name}' ran out of eligible patients entirely "
                              f"at prototype slot {k+1}/{n_prototypes_this_class} -- skipping this slot.")
                    continue

                eligible_z = z_norm[eligible_mask]
                eligible_idxs = torch.where(eligible_mask)[0]
                sims = scale * (p_block[k:k+1] @ eligible_z.T).squeeze(0)
                best_local = sims.argmax().item()
                train_idx = int(eligible_idxs[best_local].item())
                best_sim = float(sims[best_local].item())

                model.proto_layer.prototypes[p_idx].copy_(all_z[train_idx])
                proj[p_idx] = {"train_idx": train_idx, "patient_id": str(pid_train[train_idx]),
                              "class_idx": class_idx, "class_name": name,
                              "sim_score": best_sim,
                              "used_comorbid_fallback": not bool(purity_mask[train_idx].item())}
                used_mask[train_idx] = True
    if verbose:
        n_fallback = sum(1 for v in proj.values() if v["used_comorbid_fallback"])
        print(f"  projected {len(proj)} prototypes ({n_fallback} used the comorbid fallback)")
    return proj


def run_one_epoch(model, loader, optimizer, C_jac, class_weights, cfg, device, train=True):
    model.train(train)
    totals = {k: torch.zeros((), device=device) for k in
             ["total", "bce", "clst", "sep", "div", "cntrst", "covar"]}
    n_batches = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for X_b, Y_b in loader:
            X_b, Y_b = X_b.to(device, non_blocking=True), Y_b.to(device, non_blocking=True)
            logits, sim, z = model(X_b)
            l_bce = bce_loss(logits, Y_b, class_weights)
            l_clst = clustering_loss(sim, Y_b, model.proto_layer.assignment)
            l_sep = separation_loss(sim, Y_b, model.proto_layer.assignment)
            l_div = orthogonality_loss(model.proto_layer.prototypes)
            l_cntrst = contrastive_loss(model.proto_layer.prototypes, C_jac)
            l_covar = covariance_loss(z)
            l_total = (l_bce + cfg["lambda_clst"]*l_clst + cfg["lambda_sep"]*l_sep
                       + cfg["lambda_div"]*l_div + cfg["lambda_cntrst"]*l_cntrst
                       + cfg.get("lambda_covar", 0.0)*l_covar)
            if train:
                optimizer.zero_grad(set_to_none=True)
                l_total.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            totals["total"] += l_total.detach(); totals["bce"] += l_bce.detach()
            totals["clst"] += l_clst.detach(); totals["sep"] += l_sep.detach()
            totals["div"] += l_div.detach(); totals["cntrst"] += l_cntrst.detach()
            totals["covar"] += l_covar.detach()
            n_batches += 1
    return {k: (v / max(n_batches, 1)).item() for k, v in totals.items()}


def train_protoehr(cfg, data, device, eval_fn, verbose=True, branch_name="", trial=None):
    torch.manual_seed(cfg["random_seed"]); np.random.seed(cfg["random_seed"])
    X_train, Y_train, pid_train = data["X_train"], data["Y_train"], data["pid_train"]
    X_val, Y_val, pid_val = data["X_val"], data["Y_val"], data["pid_val"]
    class_weights = data["class_weights"].to(device)

    train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(Y_train, dtype=torch.float32))
    train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True, drop_last=True)

    scale = cfg["adapter_out"] ** 0.5
    model = ProtoEHRNet(cfg, scale, verbose=verbose).to(device)
    C_jac = build_jaccard_matrix(Y_train, cfg["prototypes_per_class"], cfg["zero_diagonal_blocks"]).to(device)

    best_iter_metric, iter_patience_ctr = -1.0, 0
    current_lr = cfg["lr"]
    overall_best_state = None
    projection_map = {}

    for iteration in range(1, cfg["max_iterations"] + 1):
        optimizer = torch.optim.Adam(model.parameters(), lr=current_lr, weight_decay=cfg["weight_decay"])
        best_epoch_metric, epoch_patience, best_epoch_state = -1.0, 0, None

        for epoch in range(1, cfg["max_epochs"] + 1):
            run_one_epoch(model, train_loader, optimizer, C_jac, class_weights, cfg, device, train=True)
            va = eval_fn(model, X_val, Y_val, pid_val, device)
            primary = va["_primary"]
            improved = (not np.isnan(primary)) and (primary > best_epoch_metric)
            if improved or epoch == 1:
                if improved:
                    best_epoch_metric = primary; epoch_patience = 0
                best_epoch_state = {k: v.clone() for k, v in model.state_dict().items()}
                if not improved:
                    epoch_patience += 1
            else:
                epoch_patience += 1
            if epoch_patience >= cfg["patience"]:
                break

        model.load_state_dict(best_epoch_state)
        if best_epoch_metric > best_iter_metric:
            best_iter_metric = best_epoch_metric; iter_patience_ctr = 0
            overall_best_state = copy.deepcopy(best_epoch_state)
        else:
            iter_patience_ctr += 1

        projection_map = project_prototypes(model, X_train, Y_train, pid_train, device, verbose=verbose)
        if verbose:
            print(f"  [{branch_name}] iter {iteration}/{cfg['max_iterations']} "
                  f"best={best_epoch_metric:.4f} all-time={best_iter_metric:.4f} "
                  f"iter_pat={iter_patience_ctr}/{cfg['iter_patience']}")

        if trial is not None:
            trial.report(best_iter_metric, step=iteration)
            if trial.should_prune():
                raise optuna.TrialPruned()

        if iter_patience_ctr >= cfg["iter_patience"]:
            model.load_state_dict(overall_best_state)
            projection_map = project_prototypes(model, X_train, Y_train, pid_train, device, verbose=verbose)
            break
        current_lr *= cfg["lr_decay"]

    model.load_state_dict(overall_best_state)
    final_metrics = eval_fn(model, X_val, Y_val, pid_val, device)
    proto_diag = prototype_diagnostics(model)
    return model, final_metrics, proto_diag, projection_map


In [0]:

import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

def print_progress_callback(study, trial, every=5):
    if (trial.number + 1) % every == 0:
        print(f"    [{study.study_name}] trial {trial.number + 1} done -- "
              f"best so far: {study.best_value:.4f}")


LR_GRID = [5e-4, 1e-3, 1.5e-3, 2e-3, 2.5e-3, 3e-3]


def make_eval_fn(branch, k_pool=5):
    if branch == "global":
        return lambda model, X, Y, pid, device: evaluate_all(model, X, Y, device)
    else:
        return lambda model, X, Y, pid, device: evaluate_patient_level(model, X, Y, pid, device, k=k_pool)


def sample_phase_a(trial, base_cfg, min_k=1, max_k=20):
    cfg = copy.deepcopy(base_cfg)
    cfg["prototypes_per_class"] = {
        "control": trial.suggest_int("k_control", min_k, max_k),
        "pe_ghtn": trial.suggest_int("k_pe_ghtn", min_k, max_k),
        "gdm":     trial.suggest_int("k_gdm", min_k, max_k),
    }
    cfg["run_id"] = f"phaseA_trial_{trial.number}"
    return cfg


def sample_phase_b(trial, base_cfg):
    cfg = copy.deepcopy(base_cfg)
    cfg["lambda_clst"]   = trial.suggest_float("lambda_clst", 1e-4, 1e-1, log=True)
    cfg["lambda_sep"]    = trial.suggest_float("lambda_sep", 1e-5, 1e-2, log=True)
    cfg["lambda_div"]    = trial.suggest_float("lambda_div", 1e-2, 50.0, log=True)
    cfg["lambda_cntrst"] = trial.suggest_float("lambda_cntrst", 1.0, 500.0, log=True)
    cfg["lambda_covar"]  = trial.suggest_float("lambda_covar", 1e-2, 20.0, log=True)
    cfg["lr"]            = trial.suggest_categorical("lr", LR_GRID)
    cfg["run_id"]        = f"phaseB_trial_{trial.number}"
    return cfg


def make_objective(data, device, branch, sample_fn, base_cfg, hpo_epochs=60, hpo_iterations=3, **sample_kwargs):
    eval_fn = make_eval_fn(branch, k_pool=base_cfg.get("topk_pool_k", 5))

    def objective(trial):
        cfg = sample_fn(trial, base_cfg, **sample_kwargs)
        cfg["max_epochs"], cfg["max_iterations"] = hpo_epochs, hpo_iterations
        cfg["_branch_name"] = branch
        try:
            model, final_metrics, proto_diag, projection_map = train_protoehr(
                cfg, data, device, eval_fn, verbose=False, branch_name=branch, trial=trial
            )
        except optuna.TrialPruned:
            raise
        except Exception as e:
            print(f"[{branch} trial {trial.number}] FAILED: {e}")
            return 0.0

        # OPTIMIZE ON auroc_macro ONLY -- print every other macro metric
        # for visibility, but only auroc_macro drives Optuna's search.
        print(f"[{branch} trial {trial.number}] "
              f"auroc_macro={final_metrics['auroc_macro']:.4f} (OPTIMIZED)  |  "
              f"auprc_macro={final_metrics['auprc_macro']:.4f}  "
              f"f1_macro={final_metrics['f1_macro']:.4f}  "
              f"bacc_macro={final_metrics['bacc_macro']:.4f}  "
              f"brier_macro={final_metrics['brier_macro']:.4f}")

        trial.set_user_attr("final_metrics", final_metrics)
        trial.set_user_attr("proto_diag", proto_diag)
        trial.set_user_attr("cfg", cfg)
        return final_metrics["auroc_macro"]
    return objective

def _enqueue_baseline_phase_a(study, base_cfg):
    study.enqueue_trial({
        "k_control": base_cfg["prototypes_per_class"]["control"],
        "k_pe_ghtn": base_cfg["prototypes_per_class"]["pe_ghtn"],
        "k_gdm":     base_cfg["prototypes_per_class"]["gdm"],
    })


def _enqueue_baseline_phase_b(study, base_cfg):
    study.enqueue_trial({
        "lambda_clst":   base_cfg["lambda_clst"],
        "lambda_sep":    base_cfg["lambda_sep"],
        "lambda_div":    base_cfg["lambda_div"],
        "lambda_cntrst": base_cfg["lambda_cntrst"],
        "lambda_covar":  base_cfg["lambda_covar"],
        "lr":            base_cfg["lr"],
    })


def run_two_phase_hpo(branch, data, device, n_trials_a=100, n_trials_b=300,
                      hpo_epochs=60, hpo_iterations=3, min_k=1, max_k=20,
                      base_cfg=None, seed=42, verbose=True):
    base_cfg = base_cfg or (GLOBAL_CFG if branch == "global" else LOCAL_CFG)

    pruner = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1)
    sampler_a = optuna.samplers.TPESampler(seed=seed)
    study_a = optuna.create_study(direction="maximize", sampler=sampler_a, pruner=pruner,
                                  study_name=f"{branch}_phaseA")
    _enqueue_baseline_phase_a(study_a, base_cfg)
    study_a.optimize(make_objective(data, device, branch, sample_phase_a, base_cfg,
                                    hpo_epochs, hpo_iterations, min_k=min_k, max_k=max_k),
                     n_trials=n_trials_a, show_progress_bar=False, callbacks=[print_progress_callback])
    best_a = study_a.best_trial
    if verbose:
        print(f"[{branch}] Phase A best trial #{best_a.number}: {best_a.value:.4f}")
        print(f"  k_control={best_a.params['k_control']} k_pe_ghtn={best_a.params['k_pe_ghtn']} "
              f"k_gdm={best_a.params['k_gdm']}")

    phase_b_base = copy.deepcopy(base_cfg)
    phase_b_base["prototypes_per_class"] = {
        "control": best_a.params["k_control"],
        "pe_ghtn": best_a.params["k_pe_ghtn"],
        "gdm":     best_a.params["k_gdm"],
    }

    sampler_b = optuna.samplers.TPESampler(seed=seed)
    study_b = optuna.create_study(direction="maximize", sampler=sampler_b, pruner=pruner,
                                  study_name=f"{branch}_phaseB")
    _enqueue_baseline_phase_b(study_b, base_cfg)
    study_b.optimize(make_objective(data, device, branch, sample_phase_b, phase_b_base,
                                    hpo_epochs, hpo_iterations),
                     n_trials=n_trials_b, show_progress_bar=False, callbacks=[print_progress_callback])
    best_b = study_b.best_trial
    if verbose:
        print(f"[{branch}] Phase B best trial #{best_b.number}: {best_b.value:.4f}")

    final_cfg = copy.deepcopy(best_b.user_attrs["cfg"])
    final_cfg["max_epochs"], final_cfg["max_iterations"] = 200, 5
    final_cfg["run_id"] = f"{branch}_final"
    return {"branch": branch, "study_a": study_a, "study_b": study_b,
           "best_a": best_a, "best_b": best_b, "final_cfg": final_cfg}


## Fusion helper definitions (PerPrototypeFusion w/ age covariate, z-score norm, similarity extraction)

In [0]:
def get_patient_level_similarities(model, X, pid, device, k=None):
    """Returns (patient_ids, sim_matrix) -- top-k pooled if k is given
    (local branch), raw row-level otherwise (global branch, already one
    row per patient)."""
    if k is not None:
        pooled_pid, _, pooled_sim = topk_pool_patient_probs(model, X, pid, device, k=k)
        return pooled_pid, pooled_sim
    model.eval()
    with torch.no_grad():
        all_sim = []
        for start in range(0, len(X), 512):
            xb = torch.tensor(X[start:start+512], dtype=torch.float32).to(device)
            z = model.adapter(xb)
            all_sim.append(model.proto_layer(z).cpu().numpy())
    return np.asarray(pid), np.concatenate(all_sim, axis=0)


def zscore_fit(X):
    mean, std = X.mean(axis=0, keepdims=True), X.std(axis=0, keepdims=True)
    return mean, std.clip(min=1e-8)

def zscore_apply(X, mean, std):
    return (X - mean) / std


class ConcatFusion(nn.Module):
    """Replaces prototype-pairing-based fusion with simple concatenation
    of BOTH branches' full similarity vectors"""
    def __init__(self, n_local_prototypes, n_global_prototypes, n_classes, n_covariates=1):
        super().__init__()
        self.n_local_prototypes = n_local_prototypes
        self.n_global_prototypes = n_global_prototypes
        self.classifier = nn.Linear(n_local_prototypes + n_global_prototypes + n_covariates, n_classes)

    def forward(self, sim_local, sim_global, covariates):
        combined = torch.cat([sim_local, sim_global, covariates], dim=1)
        return self.classifier(combined)


def train_fusion_layer(sim_local_train, sim_global_train, age_train, Y_train,
                       sim_local_val, sim_global_val, age_val, Y_val,
                       n_local_prototypes, n_global_prototypes, n_classes, class_weights, device,
                       lr=1e-3, weight_decay=1e-4, lambda_l1=0.0,
                       max_epochs=200, patience=15, batch_size=512, verbose=True):
    """lambda_l1 penalizes the L1 norm of ONLY the classifier's weight
    matrix (not the bias) -- with concatenated, unpaired inputs, this is
    now the primary way to read per-prototype importance: a near-zero
    weight column for a given local or global prototype means it isn't
    meaningfully driving any class's prediction, independent of any
    forced pairing with a prototype from the other branch."""
    model = ConcatFusion(n_local_prototypes, n_global_prototypes, n_classes, n_covariates=1).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=class_weights)

    age_train_n = (age_train - age_train.mean()) / (age_train.std() + 1e-8)
    age_val_n = (age_val - age_train.mean()) / (age_train.std() + 1e-8)

    ds = TensorDataset(
        torch.tensor(sim_local_train, dtype=torch.float32),
        torch.tensor(sim_global_train, dtype=torch.float32),
        torch.tensor(age_train_n.reshape(-1, 1), dtype=torch.float32),
        torch.tensor(Y_train, dtype=torch.float32),
    )
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)

    Xl_val = torch.tensor(sim_local_val, dtype=torch.float32).to(device)
    Xg_val = torch.tensor(sim_global_val, dtype=torch.float32).to(device)
    age_val_t = torch.tensor(age_val_n.reshape(-1, 1), dtype=torch.float32).to(device)

    best_auroc, best_state, patience_ctr = -1.0, None, 0
    for epoch in range(1, max_epochs + 1):
        model.train()
        for xl, xg, a, y in loader:
            xl, xg, a, y = xl.to(device), xg.to(device), a.to(device), y.to(device)
            logits = model(xl, xg, a)
            loss = loss_fn(logits, y)
            if lambda_l1 > 0:
                loss = loss + lambda_l1 * model.classifier.weight.abs().sum()
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xl_val, Xg_val, age_val_t)
            p_val = torch.sigmoid(val_logits).cpu().numpy()
        results = _compute_metric_suite(Y_val, p_val)
        macro_auroc = results["auroc_macro"]

        if macro_auroc > best_auroc:
            best_auroc = macro_auroc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
        if verbose and (epoch == 1 or epoch % 10 == 0):
            print(f"  fusion ep {epoch:3d} | val macro AUROC {macro_auroc:.4f} | pat {patience_ctr}/{patience}")
        if patience_ctr >= patience:
            break

    model.load_state_dict(best_state)
    return model, best_auroc

## Step 1 — paste configs, retrain + freeze both branches

In [0]:
# ============================================================
# FUSION -- paste the two printed final_cfg dicts below, then run this
# cell. Retrains both branches at full budget, freezes them, fuses
# (z-score norm + per-prototype weight + age covariate + L1), evaluates
# on held-out test ONCE.
# ============================================================
FINAL_CFG_GLOBAL = {'flags_path': 'catalog.schema.step7_filtered_cohort_with_apos', 'output_dir': './proto_runs', 'embedding_dim': 768, 'adapter_hidden': 256, 'adapter_out': 64, 'n_classes': 3, 'prototypes_per_class': {'control': 17, 'pe_ghtn': 9, 'gdm': 9}, 'lr': 0.0005, 'weight_decay': 0.0001, 'batch_size': 512, 'max_epochs': 200, 'patience': 10, 'lambda_clst': 0.00045845413117644694, 'lambda_sep': 0.002298951753515831, 'lambda_div': 49.337245549499, 'lambda_cntrst': 66.98658887574608, 'lambda_covar': 0.0579582453330533, 'zero_diagonal_blocks': True, 'use_jscl_pretrain': False, 'max_iterations': 5, 'iter_patience': 2, 'lr_decay': 0.5, 'random_seed': 42, 'run_id': 'global_final', 'embeddings_path': 'catalog.schema.patient_clmbr_embeddings_new_v5', '_branch_name': 'global'}
FINAL_CFG_LOCAL  = {'flags_path': 'catalog.schema.step7_filtered_cohort_with_apos', 'output_dir': './proto_runs', 'embedding_dim': 768, 'adapter_hidden': 256, 'adapter_out': 64, 'n_classes': 3, 'prototypes_per_class': {'control': 6, 'pe_ghtn': 5, 'gdm': 18}, 'lr': 0.0005, 'weight_decay': 0.0001, 'batch_size': 512, 'max_epochs': 200, 'patience': 10, 'lambda_clst': 0.007368247757287483, 'lambda_sep': 0.006551162993545934, 'lambda_div': 0.4195993256985269, 'lambda_cntrst': 65.47464665971741, 'lambda_covar': 2.658175767843088, 'zero_diagonal_blocks': True, 'use_jscl_pretrain': False, 'max_iterations': 5, 'iter_patience': 2, 'lr_decay': 0.5, 'random_seed': 42, 'run_id': 'local_final', 'embeddings_path': 'catalog.schema.patient_clmbr_embeddings_new_v5_rolling_trimester_delta', 'topk_pool_k': 5, 'max_train_patients_per_stratum': None, '_branch_name': 'local'}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
split = build_person_split(BASE_CFG)  # SAME split, deterministic given same BASE_CFG/seed
global_data = load_global_data(GLOBAL_CFG, split, device)
local_data = load_local_data(LOCAL_CFG, split, device)  # FULL data, not an HPO subsample

global_eval_fn = make_eval_fn("global")
local_eval_fn = make_eval_fn("local", k_pool=LOCAL_CFG.get("topk_pool_k", 5))

global_model, global_val_metrics, global_proto_diag, global_projection_map = train_protoehr(
    FINAL_CFG_GLOBAL, global_data, device, global_eval_fn, verbose=True, branch_name="global"
)
local_model, local_val_metrics, local_proto_diag, local_projection_map = train_protoehr(
    FINAL_CFG_LOCAL, local_data, device, local_eval_fn, verbose=True, branch_name="local"
)

for p in global_model.parameters():
    p.requires_grad_(False)
for p in local_model.parameters():
    p.requires_grad_(False)
global_model.eval(); local_model.eval()

print("\n=== Global branch (frozen) validation metrics ===")
for k, v in global_val_metrics.items():
    if not k.startswith("_"):
        print(f"  {k:<32} {v:.4f}")
print("\n=== Local branch (frozen) validation metrics ===")
for k, v in local_val_metrics.items():
    if not k.startswith("_"):
        print(f"  {k:<32} {v:.4f}")


## Step 2 — fusion (z-score norm, per-prototype weight, age covariate, L1 sweep)

In [0]:
k_pool = LOCAL_CFG.get("topk_pool_k", 5)

def build_fusion_split(global_data, local_data, split_name, device, k_pool):
    """No more align_by_class_block -- local and global similarity
    vectors are kept at their OWN full width, no truncation/pairing."""
    g_pid, g_sim = get_patient_level_similarities(
        global_model, global_data[f"X_{split_name}"], global_data[f"pid_{split_name}"], device, k=None)
    l_pid, l_sim = get_patient_level_similarities(
        local_model, local_data[f"X_{split_name}"], local_data[f"pid_{split_name}"], device, k=k_pool)
    common = sorted(set(g_pid) & set(l_pid))
    g_idx = {pid: i for i, pid in enumerate(g_pid)}
    l_idx = {pid: i for i, pid in enumerate(l_pid)}
    g_rows = np.array([g_idx[p] for p in common])
    l_rows = np.array([l_idx[p] for p in common])
    y_lookup = pd.DataFrame(global_data[f"Y_{split_name}"], index=global_data[f"pid_{split_name}"])
    age_lookup = pd.Series(global_data[f"age_{split_name}"], index=global_data[f"pid_{split_name}"])
    Y = y_lookup.loc[common].values.astype(np.float32)
    age = age_lookup.loc[common].values.astype(np.float32)
    return common, l_sim[l_rows], g_sim[g_rows], Y, age


pids_train, l_sim_train, g_sim_train, Y_fusion_train, age_train = build_fusion_split(
    global_data, local_data, "train", device, k_pool)
pids_val, l_sim_val, g_sim_val, Y_fusion_val, age_val = build_fusion_split(
    global_data, local_data, "val", device, k_pool)
pids_test, l_sim_test, g_sim_test, Y_fusion_test, age_test = build_fusion_split(
    global_data, local_data, "test", device, k_pool)

print(f"Fusion patients: train={len(pids_train)} val={len(pids_val)} test={len(pids_test)}")
print(f"local sim dim: {l_sim_train.shape[1]}, global sim dim: {g_sim_train.shape[1]} "
      f"(no truncation -- full width of each branch used)")

l_mean, l_std = zscore_fit(l_sim_train)
g_mean, g_std = zscore_fit(g_sim_train)
l_sim_train_n = zscore_apply(l_sim_train, l_mean, l_std)
g_sim_train_n = zscore_apply(g_sim_train, g_mean, g_std)
l_sim_val_n = zscore_apply(l_sim_val, l_mean, l_std)
g_sim_val_n = zscore_apply(g_sim_val, g_mean, g_std)
l_sim_test_n = zscore_apply(l_sim_test, l_mean, l_std)
g_sim_test_n = zscore_apply(g_sim_test, g_mean, g_std)

n_local_prototypes = l_sim_train_n.shape[1]
n_global_prototypes = g_sim_train_n.shape[1]
pos_freq = Y_fusion_train.mean(axis=0).clip(0.03, 0.97)
fusion_class_weights = torch.tensor((1 - pos_freq) / pos_freq, dtype=torch.float32).to(device)

fusion_sampler = optuna.samplers.TPESampler(seed=42)
fusion_study = optuna.create_study(direction="maximize", sampler=fusion_sampler, study_name="fusion_l1")

def fusion_objective(trial):
    lambda_l1 = trial.suggest_float("lambda_l1", 1e-6, 1e-1, log=True) if trial.number > 0 else 0.0
    _, val_auroc = train_fusion_layer(
        l_sim_train_n, g_sim_train_n, age_train, Y_fusion_train,
        l_sim_val_n, g_sim_val_n, age_val, Y_fusion_val,
        n_local_prototypes=n_local_prototypes, n_global_prototypes=n_global_prototypes,
        n_classes=len(CLASS_NAMES), class_weights=fusion_class_weights, device=device,
        lambda_l1=lambda_l1, verbose=False,
    )
    trial.set_user_attr("lambda_l1", lambda_l1)
    return val_auroc

fusion_study.optimize(fusion_objective, n_trials=20, show_progress_bar=False, callbacks=[print_progress_callback])
best_lambda_l1 = fusion_study.best_trial.user_attrs["lambda_l1"]
print(f"\nBest fusion lambda_l1: {best_lambda_l1:.6g}  (val macro AUROC {fusion_study.best_value:.4f})")

fusion_model, fusion_val_auroc = train_fusion_layer(
    l_sim_train_n, g_sim_train_n, age_train, Y_fusion_train,
    l_sim_val_n, g_sim_val_n, age_val, Y_fusion_val,
    n_local_prototypes=n_local_prototypes, n_global_prototypes=n_global_prototypes,
    n_classes=len(CLASS_NAMES), class_weights=fusion_class_weights, device=device,
    lambda_l1=best_lambda_l1,
)

print(f"\nFusion model -- final val macro AUROC: {fusion_val_auroc:.4f}")

with torch.no_grad():
    W = fusion_model.classifier.weight.cpu().numpy()  # (n_classes, n_local+n_global+1)
w_local = W[:, :n_local_prototypes]
w_global = W[:, n_local_prototypes:n_local_prototypes + n_global_prototypes]
w_age = W[:, -1]

print("\n=== Local prototype weights (per class, then mean |weight| summary) ===")
header = "  proto  " + "".join(f"{name:>12s}" for name in CLASS_NAMES) + f"{'mean|w|':>12s}"
print(header)
for i in range(n_local_prototypes):
    row = f"  {i:5d}  " + "".join(f"{w_local[c, i]:+12.4f}" for c in range(len(CLASS_NAMES)))
    row += f"{np.abs(w_local[:, i]).mean():12.4f}"
    print(row)

print("\n=== Global prototype weights (per class, then mean |weight| summary) ===")
print(header)
for i in range(n_global_prototypes):
    row = f"  {i:5d}  " + "".join(f"{w_global[c, i]:+12.4f}" for c in range(len(CLASS_NAMES)))
    row += f"{np.abs(w_global[:, i]).mean():12.4f}"
    print(row)

print("\n=== Age covariate weight (per class) ===")
for c, name in enumerate(CLASS_NAMES):
    print(f"  {name:10s}: {w_age[c]:+.4f}")

## Step 3 — final held-out test evaluation (touched once)

In [0]:
fusion_model.eval()
age_test_n = (age_test - age_train.mean()) / (age_train.std() + 1e-8)

with torch.no_grad():
    Xl_test_t = torch.tensor(l_sim_test_n, dtype=torch.float32).to(device)
    Xg_test_t = torch.tensor(g_sim_test_n, dtype=torch.float32).to(device)
    age_test_t = torch.tensor(age_test_n.reshape(-1, 1), dtype=torch.float32).to(device)
    test_logits = fusion_model(Xl_test_t, Xg_test_t, age_test_t)  # no more test_alpha
    p_test = torch.sigmoid(test_logits).cpu().numpy()

test_metrics = _compute_metric_suite(Y_fusion_test, p_test)

print("=== FINAL HELD-OUT TEST METRICS (fused model, touched once) ===")
for key in ["auroc_macro", "auprc_macro", "f1_macro", "bacc_macro", "brier_macro",
           "auroc_macro_outcomes_only", "auprc_macro_outcomes_only"]:
    print(f"  {key:<32} {test_metrics[key]:.4f}")

print("\nPer-class test metrics:")
for name in CLASS_NAMES:
    print(f"  {name:10s}  AUROC {test_metrics[f'auroc_{name}']:.4f}  "
          f"AUPRC {test_metrics[f'auprc_{name}']:.4f}  F1 {test_metrics[f'f1_{name}']:.4f}  "
          f"BalAcc {test_metrics[f'bacc_{name}']:.4f}  Brier {test_metrics[f'brier_{name}']:.4f}")


# ============================================================================
# Bootstrapped 95% CIs
# ============================================================================
def bootstrap_fused_test_metrics_ci(Y_true, Y_prob, class_names, n_bootstrap=1000,
                                    confidence=0.95, seed=42):
    rng = np.random.RandomState(seed)
    n = Y_true.shape[0]
    macro_keys = ["auroc_macro", "auprc_macro", "f1_macro", "bacc_macro", "brier_macro",
                 "auroc_macro_outcomes_only", "auprc_macro_outcomes_only"]
    per_class_keys = [f"{m}_{name}" for m in ["auroc", "auprc", "f1", "bacc", "brier"] for name in class_names]
    all_keys = macro_keys + per_class_keys
    boot_scores = {k: [] for k in all_keys}

    n_skipped = 0
    for _ in range(n_bootstrap):
        idx = rng.randint(0, n, n)
        yt_r, yp_r = Y_true[idx], Y_prob[idx]
        degenerate = any(yt_r[:, c].sum() == 0 or yt_r[:, c].sum() == n for c in range(len(class_names)))
        if degenerate:
            n_skipped += 1
            continue
        m = _compute_metric_suite(yt_r, yp_r, class_names=class_names, outcome_names=OUTCOME_NAMES)
        for k in all_keys:
            boot_scores[k].append(m[k])

    results = {}
    for k in all_keys:
        arr = np.array(boot_scores[k])
        arr = arr[~np.isnan(arr)] 
        if len(arr) == 0:
            results[k] = (float("nan"), float("nan"), float("nan"))
            continue
        lo, hi = np.percentile(arr, [(1-confidence)/2*100, (1+confidence)/2*100])
        results[k] = (arr.mean(), lo, hi)

    if n_skipped > 0:
        print(f"  ({n_skipped}/{n_bootstrap} resamples skipped -- degenerate label in that draw)")
    return results


print("\n" + "="*70)
print("BOOTSTRAPPED 95% CIs (1000 resamples, held-out test)")
print("="*70)
fused_ci = bootstrap_fused_test_metrics_ci(Y_fusion_test, p_test, CLASS_NAMES)

print("\nMacro metrics:")
for key in ["auroc_macro", "auprc_macro", "f1_macro", "bacc_macro", "brier_macro",
           "auroc_macro_outcomes_only", "auprc_macro_outcomes_only"]:
    mean, lo, hi = fused_ci[key]
    print(f"  {key:<32} {mean:.4f}  (95% CI [{lo:.4f}, {hi:.4f}])")

print("\nPer-class metrics:")
for name in CLASS_NAMES:
    print(f"  {name}:")
    for m in ["auroc", "auprc", "f1", "bacc", "brier"]:
        mean, lo, hi = fused_ci[f"{m}_{name}"]
        print(f"    {m:8s} {mean:.4f}  (95% CI [{lo:.4f}, {hi:.4f}])")

In [0]:
# ============================================================================
# Step 5 -- Ablations: Global Only / Global + Fusion Layer / Local Only / Local + Fusion Layer
# (+ the full fused model restated for reference). Every variant is scored on the SAME population
# (pids_test) AND its test probabilities are stored in pids_test order, so the saved arrays line up
# row-for-row with Y_fusion_test for a paired significance test later.
#   native_global            = global_model's own trained classifier (no fusion machinery)
#   native_local             = local_model's own classifier with top-k pooling, restricted to pids_test
#   global_only_fusion_style = FRESH fusion-style classifier on global prototypes only (local width 0)
#   local_only_fusion_style  = FRESH fusion-style classifier on local prototypes only (global width 0)
# ============================================================================

def train_single_branch_classifier(sim_train_n, age_train, Y_train,
                                   sim_val_n, age_val, Y_val,
                                   n_prototypes, n_classes, class_weights, device,
                                   branch_label, n_l1_trials=20, seed=42):
    """Reuses ConcatFusion/train_fusion_layer with the OTHER branch's width set to 0 (an empty (n, 0) array);
    torch.cat handles a zero-width tensor, so this is the exact full-fusion training procedure on one branch."""
    zeros_train = np.zeros((len(sim_train_n), 0), dtype=np.float32)
    zeros_val = np.zeros((len(sim_val_n), 0), dtype=np.float32)

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=seed),
                                study_name=f"{branch_label}_only_l1")

    def objective(trial):
        lambda_l1 = trial.suggest_float("lambda_l1", 1e-6, 1e-1, log=True) if trial.number > 0 else 0.0
        _, val_auroc = train_fusion_layer(
            sim_train_n, zeros_train, age_train, Y_train,
            sim_val_n, zeros_val, age_val, Y_val,
            n_local_prototypes=n_prototypes, n_global_prototypes=0,
            n_classes=n_classes, class_weights=class_weights, device=device,
            lambda_l1=lambda_l1, verbose=False,
        )
        trial.set_user_attr("lambda_l1", lambda_l1)
        return val_auroc

    study.optimize(objective, n_trials=n_l1_trials, show_progress_bar=False, callbacks=[print_progress_callback])
    best_lambda_l1 = study.best_trial.user_attrs["lambda_l1"]
    print(f"  [{branch_label}-only] best lambda_l1={best_lambda_l1:.6g}, val macro AUROC={study.best_value:.4f}")

    model, val_auroc = train_fusion_layer(
        sim_train_n, zeros_train, age_train, Y_train,
        sim_val_n, zeros_val, age_val, Y_val,
        n_local_prototypes=n_prototypes, n_global_prototypes=0,
        n_classes=n_classes, class_weights=class_weights, device=device,
        lambda_l1=best_lambda_l1,
    )
    return model, best_lambda_l1


def predict_single_branch_classifier(model, sim_test_n, age_test_n, device):
    """(n_test, 3) test probabilities; row order = the order of sim_test_n (pids_test order, as in Step 3)."""
    zeros_test = np.zeros((len(sim_test_n), 0), dtype=np.float32)
    with torch.no_grad():
        sim_t = torch.tensor(sim_test_n, dtype=torch.float32).to(device)
        z_t = torch.tensor(zeros_test, dtype=torch.float32).to(device)
        age_t = torch.tensor(age_test_n.reshape(-1, 1), dtype=torch.float32).to(device)
        return torch.sigmoid(model(sim_t, z_t, age_t)).cpu().numpy()


def print_metrics_row(label, m):
    print(f"  {label:<28} AUROC {m['auroc_macro']:.4f}  AUPRC {m['auprc_macro']:.4f}  "
          f"F1 {m['f1_macro']:.4f}  BalAcc {m['bacc_macro']:.4f}  Brier {m['brier_macro']:.4f}")


# ---- bookkeeping: probabilities (pids_test order) and metrics for every variant ----
ablation_results = {}   # internal key -> metric suite (kept under the same name the export cell already reads)
ablation_probs = {}     # internal key -> (n_test, 3) probabilities in pids_test order


def row_order(pid_arr):
    """Positions that put rows keyed by pid_arr (unique patient ids) into pids_test order. Vectorized; fails loudly if a test patient is missing."""
    idx = pd.Index(np.asarray(pid_arr)).get_indexer(pids_test)
    assert (idx >= 0).all(), f"{int((idx < 0).sum())} pids_test patients not found"
    return idx


def record(name, probs):
    """Metrics are recomputed from the SAVED (aligned) probabilities against Y_fusion_test, so metrics and predictions can never disagree."""
    probs = np.asarray(probs, dtype=np.float64)
    assert probs.shape == (len(pids_test), len(CLASS_NAMES)), f"{name}: probs shape {probs.shape}"
    ablation_probs[name] = probs
    ablation_results[name] = _compute_metric_suite(Y_fusion_test, probs)
    print_metrics_row(name, ablation_results[name])


age_test_n = (age_test - age_train.mean()) / (age_train.std() + 1e-8)  # same normalization as Step 3

# ---- NATIVE global classifier, restricted to pids_test ----
print("=== Native branch classifiers (no fusion machinery, own trained classifier) ===")
mask_g = np.isin(global_data["pid_test"], pids_test)
X_g, Y_g, pid_g = global_data["X_test"][mask_g], global_data["Y_test"][mask_g], global_data["pid_test"][mask_g]
ord_g = row_order(pid_g)
assert np.allclose(Y_g[ord_g], Y_fusion_test), "native global labels do not match Y_fusion_test in pids_test order"
_, p_g = evaluate_all(global_model, X_g[ord_g], Y_g[ord_g], device, return_probs=True)   # rows already in pids_test order
record("native_global", p_g)

# ---- NATIVE local classifier (top-k pooling), restricted to pids_test ----
k_pool = LOCAL_CFG.get("topk_pool_k", 5)
mask_l = np.isin(local_data["pid_test"], pids_test)
X_l, Y_l, pid_l = local_data["X_test"][mask_l], local_data["Y_test"][mask_l], local_data["pid_test"][mask_l]
print(f"  native_local restricted to {len(set(pid_l))} patients (from {len(set(local_data['pid_test']))} in local's own full test set) "
      f"to match the {len(pids_test)}-patient common population")
_, p_l, pooled_pid = evaluate_patient_level(local_model, X_l, Y_l, pid_l, device, k=k_pool, return_probs=True)
p_l = p_l[row_order(pooled_pid)]   # pooled output is in first-appearance order (groupby sort=False), NOT pids_test order -> reorder
Y_l_patient = pd.DataFrame(Y_l).assign(pid=np.asarray(pid_l)).drop_duplicates("pid").set_index("pid").loc[pids_test].values
assert np.allclose(Y_l_patient, Y_fusion_test), "native local labels do not match Y_fusion_test in pids_test order"
record("native_local", p_l)

# ---- GLOBAL-ONLY fusion-style classifier ----
print("\n=== Global-only (fresh fusion-style classifier, global prototypes only) ===")
global_only_model, global_only_lambda_l1 = train_single_branch_classifier(
    g_sim_train_n, age_train, Y_fusion_train, g_sim_val_n, age_val, Y_fusion_val,
    n_prototypes=n_global_prototypes, n_classes=len(CLASS_NAMES),
    class_weights=fusion_class_weights, device=device, branch_label="global",
)
record("global_only_fusion_style", predict_single_branch_classifier(global_only_model, g_sim_test_n, age_test_n, device))

# ---- LOCAL-ONLY fusion-style classifier ----
print("\n=== Local-only (fresh fusion-style classifier, local prototypes only) ===")
local_only_model, local_only_lambda_l1 = train_single_branch_classifier(
    l_sim_train_n, age_train, Y_fusion_train, l_sim_val_n, age_val, Y_fusion_val,
    n_prototypes=n_local_prototypes, n_classes=len(CLASS_NAMES),
    class_weights=fusion_class_weights, device=device, branch_label="local",
)
record("local_only_fusion_style", predict_single_branch_classifier(local_only_model, l_sim_test_n, age_test_n, device))

# ---- FULL FUSION, restated for reference (p_test from Step 3 is already in pids_test order) ----
record("full_fusion", p_test)
assert np.isclose(ablation_results["full_fusion"]["auroc_macro"], test_metrics["auroc_macro"]), "full_fusion metrics differ from Step 3 test_metrics"

print("\n=== Summary: all variants, same held-out test population ===")
print(f"  {'variant':<28} {'AUROC':>8} {'AUPRC':>8} {'F1':>8} {'BalAcc':>8} {'Brier':>8}")
for label, m in ablation_results.items():
    print(f"  {label:<28} {m['auroc_macro']:>8.4f} {m['auprc_macro']:>8.4f} "
          f"{m['f1_macro']:>8.4f} {m['bacc_macro']:>8.4f} {m['brier_macro']:>8.4f}")

In [0]:
# ============================================================================
# Export ONLY the ablation results (probabilities + metrics) for the separate significance script.
# Same layout as the tabular baselines export: source / class_names / person_id / Y_true / predictions (+ metrics).
# ============================================================================
import pickle, os

ABLATION_EXPORT_CFG = {
    "path": "/path/to/exports/ablation_test_predictions.pkl",
    "source": "ablation_global_local",
    # internal key -> name used in the pickle (rename here only)
    "display_names": {
        "native_global": "Global Only",
        "global_only_fusion_style": "Global + Fusion Layer",
        "native_local": "Local Only",
        "local_only_fusion_style": "Local + Fusion Layer",
        "full_fusion": "Full Fusion",      # reference copy of the full model's test probabilities; drop this line to exclude it
    },
}
disp = ABLATION_EXPORT_CFG["display_names"]

ablation_payload = {
    "source": ABLATION_EXPORT_CFG["source"],
    "class_names": CLASS_NAMES,
    "person_id": np.asarray(pids_test).astype(str),       # all arrays below follow this order
    "Y_true": np.asarray(Y_fusion_test),
    "predictions": {disp[k]: np.asarray(ablation_probs[k]) for k in disp},          # (n_test, 3) probabilities
    "metrics": {disp[k]: ablation_results[k] for k in disp},                        # point estimates on the test set
    "lambda_l1": {disp["global_only_fusion_style"]: float(global_only_lambda_l1),
                  disp["local_only_fusion_style"]: float(local_only_lambda_l1)},
}
for name, p in ablation_payload["predictions"].items():
    assert p.shape == ablation_payload["Y_true"].shape, f"{name}: {p.shape} vs Y_true {ablation_payload['Y_true'].shape}"

os.makedirs(os.path.dirname(ABLATION_EXPORT_CFG["path"]), exist_ok=True)
with open(ABLATION_EXPORT_CFG["path"], "wb") as f:
    pickle.dump(ablation_payload, f)
print(f"Exported {len(ablation_payload['person_id'])} test patients, {len(ablation_payload['predictions'])} variants "
      f"-> {ABLATION_EXPORT_CFG['path']} ({os.path.getsize(ABLATION_EXPORT_CFG['path']) / 1e6:.2f} MB)")

In [0]:
# # ============================================================================
# # Precomputed lookups -- avoids O(n) .index()/np.where() calls repeated for
# # every patient in the aggregate loop (Section 6c can iterate hundreds of
# # patients; each unindexed lookup was re-scanning the array from scratch).
# # ============================================================================
# pid_to_test_idx = {pid: i for i, pid in enumerate(pids_test)}
# global_pid_to_test_idx = {pid: i for i, pid in enumerate(global_data["pid_test"])}

In [0]:
OMOP_TABLES = {
    "condition": ("catalog.schema.condition_occurrence_cohort", "condition_concept_id", "condition_source_concept_id"),
    "procedure": ("catalog.schema.procedure_occurrence_cohort", "procedure_concept_id", "procedure_source_concept_id"),
    "drug": ("catalog.schema.drug_exposure_cohort", "drug_concept_id", "drug_concept_id"),
    "measurement": ("catalog.schema.measurement_cohort", "measurement_concept_id", "measurement_source_concept_id"),
}
CONCEPT_TABLE = "catalog.omop.concept"

_CONCEPT_NAME_CACHE = {}

def resolve_concept_names(concept_ids):
    to_fetch = [int(c) for c in set(concept_ids) if int(c) != 0 and int(c) not in _CONCEPT_NAME_CACHE]
    if to_fetch:
        id_list_sql = ", ".join(f"'{c}'" for c in to_fetch)
        rows = spark.sql(f"""
            SELECT concept_id, concept_name FROM {CONCEPT_TABLE} WHERE concept_id IN ({id_list_sql})
        """).collect()
        for r in rows:
            _CONCEPT_NAME_CACHE[int(r.concept_id)] = r.concept_name
    return {c: _CONCEPT_NAME_CACHE.get(int(c), f"Unmapped code ({c})") for c in concept_ids}


def fetch_patient_code_set(person_id):
    codes = set()
    for domain, (table, concept_col, source_col) in OMOP_TABLES.items():
        df = spark.sql(f"""
            SELECT {concept_col} AS concept_id, {source_col} AS source_concept_id
            FROM {table} WHERE person_id = '{person_id}'
        """).toPandas()
        if df.empty:
            continue
        cid = df["concept_id"].fillna(0).astype(int)
        sid = df["source_concept_id"].fillna(0).astype(int)
        resolved = cid.where(cid != 0, sid)
        for r in resolved:
            if r != 0:
                codes.add((domain, int(r)))
    return codes


_PROTOTYPE_CODE_SET_CACHE = {}

def get_prototype_code_set(p_idx, branch):
    key = (branch, p_idx)
    if key in _PROTOTYPE_CODE_SET_CACHE:
        return _PROTOTYPE_CODE_SET_CACHE[key]
    projection_map = global_projection_map if branch == "global" else local_projection_map
    proto_patient_id = projection_map[p_idx]["patient_id"]
    codes = fetch_patient_code_set(proto_patient_id)
    _PROTOTYPE_CODE_SET_CACHE[key] = codes
    return codes

In [0]:
# ============================================================================
# Full export cell
# ============================================================================
import pickle

def compute_full_similarity_matrix(branch="global"):
    if branch == "global":
        model, data = global_model, global_data
        X, pid_arr = data["X_test"], data["pid_test"]
        with torch.no_grad():
            chunks = []
            for start in range(0, len(X), 512):
                xb = torch.tensor(X[start:start+512], dtype=torch.float32).to(device)
                z = model.adapter(xb)
                chunks.append(model.proto_layer(z).cpu().numpy())
        sim_matrix = np.concatenate(chunks, axis=0)
        proto_indices = sorted(global_projection_map.keys())
        return pd.DataFrame(sim_matrix, index=pid_arr, columns=proto_indices)
    else:
        model, data = local_model, local_data
        k_pool = LOCAL_CFG.get("topk_pool_k", 5)
        X, pid_arr = data["X_test"], data["pid_test"]
        with torch.no_grad():
            chunks = []
            for start in range(0, len(X), 512):
                xb = torch.tensor(X[start:start+512], dtype=torch.float32).to(device)
                z = model.adapter(xb)
                chunks.append(model.proto_layer(z).cpu().numpy())
        row_sim = np.concatenate(chunks, axis=0)
        proto_indices = sorted(local_projection_map.keys())
        unique_pids = pd.unique(pid_arr)
        pooled = np.zeros((len(unique_pids), row_sim.shape[1]))
        for i, pid in enumerate(unique_pids):
            block = row_sim[pid_arr == pid]
            k_eff = min(k_pool, block.shape[0])
            pooled[i] = np.partition(block, -k_eff, axis=0)[-k_eff:].mean(axis=0) if k_eff > 0 else 0
        return pd.DataFrame(pooled, index=unique_pids, columns=proto_indices)


global_similarity_df = compute_full_similarity_matrix("global")
local_similarity_df = compute_full_similarity_matrix("local")
print(f"Global similarity matrix: {global_similarity_df.shape}, Local: {local_similarity_df.shape}")


def fetch_code_sets_batched(person_ids):
    codes_by_patient = {pid: set() for pid in person_ids}
    id_list_sql = ", ".join(f"'{p}'" for p in person_ids)
    for domain, (table, concept_col, source_col) in OMOP_TABLES.items():
        df = spark.sql(f"""
            SELECT person_id, {concept_col} AS concept_id, {source_col} AS source_concept_id
            FROM {table} WHERE person_id IN ({id_list_sql})
        """).toPandas()
        if df.empty:
            continue
        cid = df["concept_id"].fillna(0).astype(int)
        sid = df["source_concept_id"].fillna(0).astype(int)
        df["resolved_id"] = cid.where(cid != 0, sid)
        df = df[df["resolved_id"] != 0]
        for pid, rid in zip(df["person_id"], df["resolved_id"]):
            if pid in codes_by_patient:
                codes_by_patient[pid].add((domain, int(rid)))
    return codes_by_patient


all_prototype_patient_ids = list(set(
    [info["patient_id"] for info in global_projection_map.values()] +
    [info["patient_id"] for info in local_projection_map.values()]
))
test_patient_code_sets = fetch_code_sets_batched(pids_test)
prototype_patient_code_sets_raw = fetch_code_sets_batched(all_prototype_patient_ids)

global_prototype_code_sets = {p_idx: prototype_patient_code_sets_raw[info["patient_id"]]
                              for p_idx, info in global_projection_map.items()}
local_prototype_code_sets = {p_idx: prototype_patient_code_sets_raw[info["patient_id"]]
                             for p_idx, info in local_projection_map.items()}

print(f"Fetched code sets for {len(test_patient_code_sets)} test patients, "
     f"{len(global_prototype_code_sets)} global + {len(local_prototype_code_sets)} local prototypes")


# ---- now build and save the export ----
global_proto_train_idx = np.array([info["train_idx"] for info in global_projection_map.values()])
local_proto_train_idx = np.array([info["train_idx"] for info in local_projection_map.values()])

export_payload = {
    "global_projection_map": global_projection_map,
    "local_projection_map": local_projection_map,
    "global_similarity_df": global_similarity_df,
    "local_similarity_df": local_similarity_df,
    "test_patient_code_sets": test_patient_code_sets,
    "global_prototype_code_sets": global_prototype_code_sets,
    "local_prototype_code_sets": local_prototype_code_sets,
    "all_prototype_patient_ids": all_prototype_patient_ids,
    "pids_test": pids_test,
    "Y_fusion_test": Y_fusion_test,
    "age_test": age_test,
    "p_test": p_test,
    "test_metrics": test_metrics,
    "CLASS_NAMES": CLASS_NAMES,
    "n_local_prototypes": n_local_prototypes,
    "n_global_prototypes": n_global_prototypes,
    "fusion_classifier_weight": fusion_model.classifier.weight.detach().cpu().numpy(),
    "fusion_classifier_bias": fusion_model.classifier.bias.detach().cpu().numpy(),
    "ablation_results": ablation_results if "ablation_results" in dir() else None,
    "global_X_test": global_data["X_test"],
    "global_pid_test": global_data["pid_test"],
    "global_proto_X": global_data["X_train"][global_proto_train_idx],
    "global_proto_indices": sorted(global_projection_map.keys()),
    "local_X_test": local_data["X_test"],
    "local_pid_test": local_data["pid_test"],
    "local_proto_X": local_data["X_train"][local_proto_train_idx],
    "local_proto_indices": sorted(local_projection_map.keys()),
}

export_payload["global_adapter_state_dict"] = {k: v.cpu() for k, v in global_model.adapter.state_dict().items()}
export_payload["global_proto_layer_state_dict"] = {k: v.cpu() for k, v in global_model.proto_layer.state_dict().items()}
export_payload["global_adapter_cfg"] = {
    "embedding_dim": GLOBAL_CFG["embedding_dim"],
    "adapter_hidden": GLOBAL_CFG["adapter_hidden"],
    "adapter_out": GLOBAL_CFG["adapter_out"],
}

export_payload["local_adapter_state_dict"] = {k: v.cpu() for k, v in local_model.adapter.state_dict().items()}
export_payload["local_proto_layer_state_dict"] = {k: v.cpu() for k, v in local_model.proto_layer.state_dict().items()}
export_payload["local_adapter_cfg"] = {
    "embedding_dim": LOCAL_CFG["embedding_dim"],
    "adapter_hidden": LOCAL_CFG["adapter_hidden"],
    "adapter_out": LOCAL_CFG["adapter_out"],
}

_check_pid = global_data["pid_test"][0]
_check_proto_idx = sorted(global_projection_map.keys())[0]
export_payload["verification_check"] = {
    "patient_id": _check_pid,
    "prototype_idx": _check_proto_idx,
    "expected_similarity": float(global_similarity_df.loc[_check_pid, _check_proto_idx]),
}

BASE_CFG["output_dir"] = "/path/to/exports"
export_path = os.path.join(BASE_CFG["output_dir"], "full_analysis_export.pkl")
with open(export_path, "wb") as f:
    pickle.dump(export_payload, f)
print(f"\nExported everything to {export_path} ({os.path.getsize(export_path) / 1e6:.1f} MB)")